In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/ipo_master_raw.csv')
print(f'Raw rows: {len(df)}')
print(df.head())
print(df.columns.tolist())

Raw rows: 2400
       IPO Date Symbol                             Company Name IPO Price  \
0  Mar 12, 2026   PAYP                       PayPay Corporation    $16.00   
1  Mar 11, 2026   SUMA             SUMA Acquisition Corporation    $10.00   
2   Mar 6, 2026   MMED                      Minimed Group, Inc.    $20.00   
3   Mar 4, 2026   GLED       GalaxyEdge Acquisition Corporation    $10.00   
4   Mar 4, 2026   KCAC  Kensington Capital Acquisition Corp. VI    $10.00   

  Current   Return  year  
0  $21.14   32.13%  2012  
1   $9.98   -0.20%  2012  
2  $15.76  -21.20%  2012  
3  $10.00        -  2012  
4  $10.05    0.50%  2012  
['IPO Date', 'Symbol', 'Company Name', 'IPO Price', 'Current', 'Return', 'year']


In [3]:
df['IPO Date'] = pd.to_datetime(df['IPO Date'], errors='coerce')
df['actual_year'] = df['IPO Date'].dt.year
print(df.groupby('actual_year').size())

actual_year
2025    1488
2026     912
dtype: int64


In [4]:
import requests

headers = {'User-Agent': 'Mozilla/5.0 (academic research project, UC Davis MSBA)'}

# test a few URL patterns for 2018
urls = [
    'https://stockanalysis.com/ipos/2018/',
    'https://stockanalysis.com/ipos/statistics/2018/',
    'https://stockanalysis.com/ipos/2018/all/',
]

for url in urls:
    r = requests.get(url, headers=headers)
    print(f'{r.status_code} -- {url}')

404 -- https://stockanalysis.com/ipos/2018/
404 -- https://stockanalysis.com/ipos/statistics/2018/
404 -- https://stockanalysis.com/ipos/2018/all/


In [5]:
import requests

headers = {'User-Agent': 'Mozilla/5.0 (academic research project, UC Davis MSBA)'}

for year in [2024, 2025]:
    url = f'https://stockanalysis.com/ipos/{year}/'
    r = requests.get(url, headers=headers)
    print(f'{r.status_code} -- {year}')

200 -- 2024
200 -- 2025


In [6]:
# Written by V
import pandas as pd

df = pd.read_csv('../data/raw/ipo_master_raw.csv')
df['IPO Date'] = pd.to_datetime(df['IPO Date'], errors='coerce')

print(f'Before cleaning: {len(df)} rows')

# remove SPACs
spac_keywords = ['acquisition', 'blank check', 'spac']
mask_spac = df['Company Name'].str.lower().str.contains('|'.join(spac_keywords), na=False)
df = df[~mask_spac]
print(f'After removing SPACs: {len(df)} rows')

# remove rows with no valid IPO price
df = df[df['IPO Price'] != '-']
df = df[df['IPO Price'].notna()]
print(f'After removing no-price rows: {len(df)} rows')

# drop rows where IPO date is null
df = df[df['IPO Date'].notna()]
print(f'After removing null dates: {len(df)} rows')

# rename columns to match schema
df = df.rename(columns={
    'Symbol': 'ticker',
    'Company Name': 'company_name',
    'IPO Date': 'ipo_date',
})

df = df[['ticker','company_name', 'ipo_date', 'year']]
df.to_csv('../data/cleaned/ipo_master.csv', index=False)
print(f'\nFinal: {len(df)} rows saved to data/cleaned/ipo_master.csv')

Before cleaning: 2655 rows
After removing SPACs: 1723 rows
After removing no-price rows: 1675 rows
After removing null dates: 1675 rows

Final: 1675 rows saved to data/cleaned/ipo_master.csv


In [8]:
import yfinance as yf
import pandas as pd
import time

df = pd.read_csv('../data/cleaned/ipo_master.csv')
print(f'Starting validation: {len(df)} tickers')

missing = []

for i, row in df.iterrows():
    ticker = row['ticker']
    try:
        data = yf.download(ticker, period='1mo', progress=False, auto_adjust=True)
        if data.empty:
            missing.append(ticker)
    except Exception as e:
        missing.append(ticker)
    
    if i % 50 == 0:
        print(f'Progress: {i}/{len(df)} -- missing so far: {len(missing)}')
    
    time.sleep(0.3)

print(f'\nDone. Missing tickers: {len(missing)}')
print(missing)

Starting validation: 1675 tickers
Progress: 0/1675 -- missing so far: 0


$CIIC: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CIIC']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$HCCO: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HCCO']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$OCFT: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['OCFT']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CHPM: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CHPM']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbo

Progress: 50/1675 -- missing so far: 15


$EXPC: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['EXPC']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$STSA: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['STSA']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$SWTX: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['SWTX']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CFB: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CFB']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol 

Progress: 100/1675 -- missing so far: 36


...

Progress: 1450/1675 -- missing so far: 474
Progress: 1500/1675 -- missing so far: 474
Progress: 1550/1675 -- missing so far: 474


$CCCM: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CCCM']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CCCX: possibly delisted; no price data found  (period=1mo)

1 Failed download:
['CCCX']: possibly delisted; no price data found  (period=1mo)


Progress: 1600/1675 -- missing so far: 476
Progress: 1650/1675 -- missing so far: 476


$MTSR: possibly delisted; no price data found  (period=1mo)

1 Failed download:
['MTSR']: possibly delisted; no price data found  (period=1mo)



Done. Missing tickers: 477
['CIIC', 'HCCO', 'OCFT', 'CHPM', 'ETNB', 'JIH', 'MOHOY', 'CNTG', 'SICP', 'MCMJ', 'OYST', 'TFFP', 'VIE', 'PING', 'IGMS', 'EXPC', 'STSA', 'SWTX', 'CFB', 'CCX', 'WSG', 'FLLC', 'LVGO', 'NOVA', 'PROS', 'MDLA', 'AMK', 'PIC', 'SCPE', 'THCA', 'KRTX', 'CHNG', 'MORF', 'LINX', 'AKRO', 'BCEL', 'PRVL', 'SMMC', 'WORK', 'MWK', 'GIX', 'RTLR', 'AGBA', 'APLT', 'AXLA', 'HHR', 'LCA', 'ATIF', 'SCPL', 'MNRL', 'TPTX', 'TUFN', 'NGM', 'SILK', 'RUHN', 'GNFT', 'SWAV', 'THCB', 'DPHC', 'SOLY', 'MITO', 'AVDR', 'TCRR', 'ANCN', 'HARP', 'GMHI', 'MDJH', 'GBS', 'VII', 'MDWT', 'SCOA', 'VIRI', 'CCV', 'KINZ', 'GHVI', 'MOTV', 'PCPC', 'PTIC', 'SBTX', 'SGTX', 'HTPA', 'KNTE', 'CAP', 'HFEN', 'OZON', 'NGMS', 'RTPZ', 'PHIC', 'DGNS', 'DMYI', 'BHSE', 'ABST', 'AJAX', 'GATO', 'ABCM', 'GHLD', 'MCFE', 'MSP', 'XPOA', 'YGMZ', 'BTWN', 'EAR', 'KRBP', 'OPT', 'CDAKQ', 'IPOE', 'IPOF', 'KRON', 'LCY', 'EMPW', 'PACE', 'TPGY', 'APSG', 'ONCR', 'VYGG', 'AGC', 'IMPX', 'ORPHY', 'GRAY', 'SYTA', 'ACTC', 'NMMC', 'VTRU', 'PTVE

In [9]:
import pandas as pd

df = pd.read_csv('../data/cleaned/ipo_master.csv')

missing = ['CIIC', 'HCCO', 'OCFT', 'CHPM', 'ETNB', 'JIH', 'MOHOY', 'CNTG', 'SICP', 'MCMJ', 'OYST', 'TFFP', 'VIE', 'PING', 'IGMS', 'EXPC', 'STSA', 'SWTX', 'CFB', 'CCX', 'WSG', 'FLLC', 'LVGO', 'NOVA', 'PROS', 'MDLA', 'AMK', 'PIC', 'SCPE', 'THCA', 'KRTX', 'CHNG', 'MORF', 'LINX', 'AKRO', 'BCEL', 'PRVL', 'SMMC', 'WORK', 'MWK', 'GIX', 'RTLR', 'AGBA', 'APLT', 'AXLA', 'HHR', 'LCA', 'ATIF', 'SCPL', 'MNRL', 'TPTX', 'TUFN', 'NGM', 'SILK', 'RUHN', 'GNFT', 'SWAV', 'THCB', 'DPHC', 'SOLY', 'MITO', 'AVDR', 'TCRR', 'ANCN', 'HARP', 'GMHI', 'MDJH', 'GBS', 'VII', 'MDWT', 'SCOA', 'VIRI', 'CCV', 'KINZ', 'GHVI', 'MOTV', 'PCPC', 'PTIC', 'SBTX', 'SGTX', 'HTPA', 'KNTE', 'CAP', 'HFEN', 'OZON', 'NGMS', 'RTPZ', 'PHIC', 'DGNS', 'DMYI', 'BHSE', 'ABST', 'AJAX', 'GATO', 'ABCM', 'GHLD', 'MCFE', 'MSP', 'XPOA', 'YGMZ', 'BTWN', 'EAR', 'KRBP', 'OPT', 'CDAKQ', 'IPOE', 'IPOF', 'KRON', 'LCY', 'EMPW', 'PACE', 'TPGY', 'APSG', 'ONCR', 'VYGG', 'AGC', 'IMPX', 'ORPHY', 'GRAY', 'SYTA', 'ACTC', 'NMMC', 'VTRU', 'PTVE', 'RTP', 'SUMO', 'ENPC', 'MTCR', 'LEAP', 'TWCT', 'NSH', 'CRHC', 'HEC', 'CMLF', 'AUVIQ', 'HCDIQ', 'AONE', 'FIII', 'STPK', 'CVAC', 'DCT', 'DGNR', 'DMYD', 'FSDC', 'CMPI', 'FRLN', 'GRSV', 'OSH', 'HOL', 'PRPB', 'VSTA', 'ALVR', 'CCIV', 'INZY', 'ITOS', 'JAMF', 'PSTH', 'CELL', 'PAND', 'TIG', 'DEH', 'HPX', 'PSTX', 'BLCT', 'ACCD', 'DNB', 'AKUS', 'FUSN', 'FMTX', 'GTH', 'NUZE', 'RPTX', 'AZEK', 'GBIO', 'AMTI', 'CALT', 'DADA', 'ZI', 'NARI', 'BMRG', 'NOVS', 'GIK', 'AYLA', 'CLEU', 'GAN', 'IPOB', 'PCPL', 'IPOC', 'DMYT', 'CCXX', 'ZGYH', 'PFHD', 'CSPR', 'PPD', 'ONEM', 'FRES', 'GHIV', 'SCVX', 'DNK', 'ADRT', 'BNOX', 'HCP', 'HORI', 'GLLI', 'MTVC', 'STET', 'LGTO', 'AHI', 'BLEU', 'NETC', 'USER', 'ENCP', 'MYNA', 'LGVC', 'WBEV', 'APNC', 'CIAN', 'JUN', 'ARCK', 'HRT', 'DTC', 'PCCT', 'INFA', 'ENFN', 'GOGN', 'AXH', 'SDIG', 'FEXD', 'AVHI', 'FNA', 'AVDX', 'ISO', 'LCW', 'TCN', 'THRX', 'DMYS', 'EXAI', 'TDCX', 'ARTE', 'HCVI', 'MEKA', 'ESMT', 'SOVO', 'THRN', 'ARGU', 'HHGC', 'FORG', 'CIIG', 'DICE', 'EZFL', 'TWKS', 'FHLT', 'CENQ', 'SSBK', 'PONO', 'ELYM', 'WEBR', 'ICVX', 'MLNK', 'PWSC', 'SNPO', 'OB', 'BASE', 'INST', 'CLOE', 'BRDG', 'IMGO', 'CORS', 'FICV', 'AVTE', 'DNAA', 'DNAB', 'DNAC', 'DNAD', 'IAS', 'THCP', 'EOCW', 'ELEV', 'MFLTF', 'MIRO', 'NEUE', 'AMAM', 'CYT', 'VERV', 'CNVY', 'WKME', 'GSQB', 'LITT', 'ISAA', 'ZMENY', 'OMIC', 'DYNS', 'PSPC', 'POND', 'ORIA', 'BRIV', 'OG', 'TALS', 'BSKY', 'EDR', 'TUGC', 'AGTI', 'IMPLQ', 'KNBE', 'ZY', 'TRKAQ', 'AKYA', 'YTPG', 'ADF', 'RPHM', 'TPGS', 'VECT', 'TIOA', 'CMLT', 'ACHL', 'TWOA', 'IKNA', 'DSEY', 'LCA', 'LVTX', 'OLK', 'VZIO', 'CRZN', 'ACTD', 'DGNU', 'GGPI', 'LEGA', 'RKTA', 'GGMC', 'LDHA', 'LVRA', 'FMIV', 'NAPA', 'RACB', 'VEI', 'GAMC', 'KSI', 'OLO', 'TETC', 'RTPY', 'GTPA', 'GTPB', 'JOANQ', 'LBPH', 'RXDX', 'FHS', 'AGGR', 'DTOC', 'SBII', 'ACQR', 'SVFB', 'SVFC', 'DMYQ', 'FRSG', 'IPVF', 'IPVI', 'WPCA', 'WPCB', 'MACC', 'AMPI', 'DHBC', 'NSTC', 'NSTD', 'IBER', 'LIII', 'SBEA', 'GIIX', 'SCR', 'COLI', 'HIII', 'GSEV', 'BRPM', 'FSII', 'CCVI', 'CHAA', 'CVII', 'DBTX', 'TSIB', 'APR', 'SGFY', 'APGB', 'BPTS', 'SCOB', 'VLON', 'ENNV', 'PICC', 'GSQD', 'TBCP', 'ATC', 'LABP', 'KRNL', 'PRPC', 'TIXT', 'HMPT', 'MIT', 'NLSP', 'OCDX', 'XM', 'FSSI', 'BTNB', 'DHHC', 'HCII', 'NSTB', 'JCIC', 'OEPW', 'MYTE', 'GMBT', 'GMII', 'LEGO', 'CLAS', 'HCCC', 'HCIC', 'TBA', 'POSH', 'FINM', 'HLAH', 'PNTM', 'ENFA', 'KUKE', 'EPWR', 'GRCL', 'LHC', 'PACX', 'PRSR', 'SVFA', 'SWBK', 'AGCB', 'VCKA', 'PPGH', 'STPC', 'THRD', 'STBX', 'CHG', 'FRZA', 'PTWO', 'BRSH', None, 'SKGR', 'PFHC', 'HLVX', 'NUBI', 'MHUA', 'GENQ', 'JGGC', 'GHIX', 'CINC', 'VIGL', 'CRGX', 'RYZB', 'SPGC', 'SRM', 'LQR', 'HRYU', 'PXDT', 'JNVR', 'WRNT', 'TSBX', 'NETD', 'PWM', 'SGE', 'SLRN', 'SYT', 'SBXC', 'DIST', 'PTHR', 'BREA', 'MGOL', 'DYNX', 'SAG', 'CEP', 'BLMZ', 'ZK', 'JUNE', 'CHRO', 'CCCM', 'CCCX', 'MTSR']

df_validated = df[~df['ticker'].isin(missing)]
df_validated = df_validated.dropna(subset=['ticker'])

print(f'Before: {len(df)} rows')
print(f'After removing missing tickers: {len(df_validated)} rows')

df_validated.to_csv('../data/cleaned/ipo_master_validated.csv', index=False)
print('Saved to data/cleaned/ipo_master_validated.csv')

Before: 1675 rows
After removing missing tickers: 1198 rows
Saved to data/cleaned/ipo_master_validated.csv


In [1]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

# test with Snowflake (SNOW) - IPO'd 2020
cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

print('Keys in response:', list(data.keys()))
print('Keys in filings:', list(data['filings'].keys()))
print('')

# check recent filings
filings = data['filings']['recent']
forms = filings['form']
print(f'Total recent filings: {len(forms)}')
print(f'Unique form types: {set(forms)}')
print(f'S-1 in recent: {"S-1" in forms}')

Keys in response: ['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings']
Keys in filings: ['recent', 'files']

Total recent filings: 1019
Unique form types: {'SC 13G/A', '5', 'SC 13G', 'SCHEDULE 13G', '144/A', '144', 'D', '3', '8-K', 'S-8', 'ARS', 'DEFA14A', 'DEF 14A', '4/A', 'SCHEDULE 13G/A', 'PRE 14A', '10-K', '3/A', '4', '10-Q'}
S-1 in recent: False


In [2]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

# check the files array
files = data['filings']['files']
print(f'Number of filing files: {len(files)}')
for f in files:
    print(f)

Number of filing files: 1
{'name': 'CIK0001640147-submissions-001.json', 'filingCount': 106, 'filingFrom': '2017-04-11', 'filingTo': '2021-03-01'}


In [3]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK0001640147-submissions-001.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

forms = data['filingIndex']['form'] if 'filingIndex' in data else data['form']
print(f'Forms in older file: {set(forms)}')
print(f'S-1 found: {"S-1" in forms}')

# find the accession number
for i, form in enumerate(forms):
    if form == 'S-1':
        accessions = data['filingIndex']['accessionNumber'] if 'filingIndex' in data else data['accessionNumber']
        dates = data['filingIndex']['filingDate'] if 'filingIndex' in data else data['filingDate']
        print(f'Accession: {accessions[i]}')
        print(f'Date: {dates[i]}')
        break

Forms in older file: {'SC 13G/A', 'DRSLTR', 'UPLOAD', 'CERT', 'CORRESP', 'SC 13G', '8-A12B', 'D', '3', 'EFFECT', '8-K', 'DRS/A', 'S-1/A', 'S-1', 'S-8', 'D/A', '40-APP/A', '424B4', 'DRS', '3/A', '4', '10-Q', '40-APP'}
S-1 found: True
Accession: 0001628280-20-013010
Date: 2020-08-24


In [4]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedant17tiwari@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
}

# test with Snowflake
cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

# check older file
f = data['filings']['files'][0]
file_url = f'https://data.sec.gov/submissions/{f["name"]}'
r2 = requests.get(file_url, headers=HEADERS)
print(f'Status: {r2.status_code}')

older = r2.json()
forms = older.get('form', [])
print(f'S-1 in older file: {"S-1" in forms}')

Status: 200
S-1 in older file: True


In [5]:
# Written by V
import requests
import pandas as pd

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research yourucd@ucdavis.edu',
    'Accept-Encoding': 'gzip, deflate',
}

# load your actual data
df = pd.read_csv('../data/cleaned/ticker_cik_map.csv')
df = df[df['cik'].notna()]

# test the first ticker
row = df.iloc[0]
print(f'Testing: {row["ticker"]} -- CIK: {row["cik"]}')

cik = str(row['cik']).zfill(10)
# remove decimal if present
cik = cik.split('.')[0].zfill(10)
print(f'CIK formatted: {cik}')

url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
print(f'Status: {r.status_code}')

if r.status_code == 200:
    data = r.json()
    forms = data['filings']['recent']['form']
    print(f'S-1 in recent: {"S-1" in forms}')
    print(f'Files array length: {len(data["filings"].get("files", []))}')

Testing: INDO -- CIK: 1757840.0
CIK formatted: 0001757840
Status: 200
S-1 in recent: False
Files array length: 0


In [6]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# pick any file that exists
files = os.listdir('../data/raw/edgar/')
print(f'Total cached files: {len(files)}')

# look at the first one
ticker = files[0].replace('.html', '')
print(f'Inspecting: {ticker}')

with open(f'../data/raw/edgar/{files[0]}', 'r', encoding='utf-8', errors='ignore') as f:
    html = f.read()

soup = BeautifulSoup(html, 'lxml')
text = soup.get_text(' ')

# print first 3000 characters to see the structure
print(text[:3000])

Total cached files: 1060
Inspecting: AAPG

 
 
 
 
 
 
 
 
 
 
 SEC.gov | Home 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
      Skip to search field
     
 
      Skip to main content
     
 
 
 
 
 
 
 
 
 
 
 
 An official website of the United States government 
 Here’s how you know 
 
 
 Here’s how you know 
 
 
 
 
 
 
 
 
 
 
                Official websites use .gov
               
 
                              A  .gov  website belongs to an official government organization in the United States.
                           
 
 
 
 
 
 
 
                Secure .gov websites use HTTPS
               
     
              A  lock 
              ( Lock A locked padlock )
              or  https://  means you’ve safely connected to the .gov website. Share sensitive information only on official, secure websites.
             
 
 
 
 
 
 
 
 
 
 
 
 
 SEC homepage 
 
 
 
 
 
        Menu
         
 
 
 
 
 
 
 
 
 
 
          Close
           
 
 
 
 
 Search SEC.gov & EDGAR 
 
 
 
 
 Sea

In [7]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
sec_homepage = 0
real_s1 = 0

for fname in files[:100]:  # check first 100
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    if 'Skip to search field' in html or 'SEC homepage' in html:
        sec_homepage += 1
    else:
        real_s1 += 1

print(f'Checked 100 files')
print(f'Real S-1 documents: {real_s1}')
print(f'SEC homepage (wrong): {sec_homepage}')

Checked 100 files
Real S-1 documents: 14
SEC homepage (wrong): 86


In [8]:
# Written by V
import requests
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

HEADERS = {
    'User-Agent': 'UC Davis Research vedant17tiwari@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
}

# test with Snowflake which we know has a valid S-1
df = pd.read_csv('../data/cleaned/s1_accessions.csv')
snow = df[df['ticker'] == 'SNOW'].iloc[0]

cik_int = int(float(snow['cik']))
accession = snow['accession_number']
acc_nodash = accession.replace('-', '')

index_url = f'https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_nodash}/{accession}-index.htm'
print(f'Index URL: {index_url}')

r = requests.get(index_url, headers={**HEADERS, 'Host': 'www.sec.gov'})
print(f'Status: {r.status_code}')

soup = BeautifulSoup(r.text, 'lxml')
print('\nAll links in index:')
for link in soup.find_all('a', href=True)[:20]:
    print(link['href'])

Index URL: https://www.sec.gov/Archives/edgar/data/1640147/000162828020013010/0001628280-20-013010-index.htm
Status: 200

All links in index:
https://www.sec.gov
https://www.sec.gov
//www.sec.gov/submit-filings/about-edgar
/cgi-bin/browse-edgar?action=getcurrent
https://www.sec.gov/edgar/search-and-access
/index.htm
/edgar/searchedgar/companysearch.html
/Archives/edgar/data/1640147/000162828020013010/snowflakes-1.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit11.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit31.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit33.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit101.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit102.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit103.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit104.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit105.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit1010.htm
/Archives/edga

In [9]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
sec_homepage = 0
real_s1 = 0

for fname in files[:100]:
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    if 'Skip to search field' in html or 'SEC homepage' in html:
        sec_homepage += 1
    else:
        real_s1 += 1

print(f'Checked 100 files')
print(f'Real S-1 documents: {real_s1}')
print(f'SEC homepage (wrong): {sec_homepage}')

Checked 100 files
Real S-1 documents: 100
SEC homepage (wrong): 0


In [10]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# find a file we know has founding year info
files = os.listdir('../data/raw/edgar/')

for fname in files[10:15]:
    ticker = fname.replace('.html', '')
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    
    soup = BeautifulSoup(html, 'lxml')
    text = soup.get_text(' ')
    
    # find sentences mentioning incorporation or founding
    sentences = text.split('.')
    for s in sentences:
        if any(word in s.lower() for word in ['incorporat', 'founded', 'organized', 'formed', 'established']):
            if any(str(y) in s for y in range(2000, 2026)):
                print(f'\n{ticker}:')
                print(s.strip()[:300])
                break


ACLX:
Corporate Information  
 We were incorporated in Delaware in December 2014 under the name Encarta Therapeutics, Inc

ACRV:
Corporate Information    We
were incorporated under the laws of the State of Delaware in March 2018

ACVA:
Corporate Information    We were incorporated
in Delaware in December 2014

ACXP:
However, our annual report on  Form 10-K for the year ended December 31, 2024, filed on March 17, 2025 , and all other documents incorporated by reference into this prospectus that were filed prior to August 4, 2025, do not give effect to the Reverse Stock Split


In [11]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
for fname in files[20:25]:
    ticker = fname.replace('.html', '')
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    soup = BeautifulSoup(html, 'lxml')
    text = soup.get_text(' ')
    sentences = text.split('.')
    for s in sentences:
        if 'raised' in s.lower() or 'funding' in s.lower() or 'series' in s.lower():
            if any(str(y) in s for y in range(2010, 2026)):
                print(f'\n{ticker}: {s.strip()[:300]}')
                break


AERO: In order to facilitate the DGIEs ability to monitor compliance in the future with certain restrictions on foreign ownership set forth in
Mexican law, as described in RegulationRegulation of the Mexican Airline IndustryForeign Investment Limitations under Mexican Law and Description of Capital

AFRM: ​ 
 ​ 
 ​ 
 
 
 Fiscal Year Ended 
 
 June 30, 
 
 
 ​ 
 ​ 
 
 
 Three Months Ended 
 
 September 30, 
 
 
 ​ 
 
 
 ​ 
 ​ 
 ​ 
 
 
 2019  
 
 
 ​ 
 ​ 
 
 
 2020 
 
 
 ​ 
 ​ 
 
 
 2019  
 
 
 ​ 
 ​ 
 
 
 2020 
 
 
 ​ 
 
 
 ​ 
 ​ 
 ​ 
 
 
 (in thousands) 
 
 
 ​ 
 
 
 
 Consolidated Statements of Oper

AFYA: As of September 30, 2019, our learning tools consisted of more than 2,033 video classes, 824 book chapters, 1,270 podcasts, 1,070 summarized texts and an exam
bank of approximately 1,604 questions;  
                Assessment tool :    Broad database suite comprising approximately 85 thousand quizz

AGCC: Type 
 
   
 
 Name 
 
   
 
 Issuing  Authority/ Registration  Insti

In [12]:
# Written by V
import pandas as pd
import sqlite3

conn = sqlite3.connect('../database/ipo_tracker.db')
vc = pd.read_csv('../data/cleaned/vc_history.csv')
print(f'Loading {len(vc)} vc_history rows...')

loaded = 0
for _, row in vc.iterrows():
    try:
        conn.execute('''INSERT OR REPLACE INTO vc_history
            (ticker, founding_year, num_rounds, last_round_type,
             total_funding_usd, has_crunchbase)
            VALUES (?,?,?,?,?,?)''',
            (row['ticker'], row.get('founding_year'), row.get('num_rounds'),
             row.get('last_round_type'), row.get('total_funding_usd'), 0))
        loaded += 1
    except Exception as e:
        print(f'Error {row["ticker"]}: {e}')

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM vc_history').fetchone()[0]
print(f'vc_history table: {count} rows')
conn.close()

Loading 1041 vc_history rows...
vc_history table: 1036 rows
